Step 2 – Upload the processed training dataset

In [1]:
from google.colab import files

uploaded = files.upload()

Saving portuguese_test_processed.csv to portuguese_test_processed.csv
Saving portuguese_train_processed.csv to portuguese_train_processed.csv


Step 3 – Import libraries

In [2]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from IPython.display import display

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.dummy import DummyRegressor
from sklearn.base import clone

from sklearn.model_selection import (
    KFold,
    cross_validate,
    cross_val_predict
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Libraries imported successfully!")

Libraries imported successfully!


Step 4 – Create output folders

In [3]:
project_dir = Path("/content/Portuguese_ML_Project")

comparison_dir = (
    project_dir / "output" / "final_model_comparison"
)

models_dir = project_dir / "models"

comparison_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

print("Comparison directory:", comparison_dir)
print("Models directory:", models_dir)

Comparison directory: /content/Portuguese_ML_Project/output/final_model_comparison
Models directory: /content/Portuguese_ML_Project/models


Step 5 – Load and validate the training dataset

In [5]:
train_df = pd.read_csv(
    "/content/portuguese_train_processed.csv"
)

X_all = train_df.drop(columns=["G3"])
y_train = train_df["G3"]

grade_features = ["num__G1", "num__G2"]

assert all(
    feature in X_all.columns
    for feature in grade_features
)

assert X_all.notna().all().all()
assert y_train.notna().all()

print("Training records:", len(y_train))
print("All features:", X_all.shape[1])
print("Linear Regression features:", grade_features)

display(train_df.head())

Training records: 519
All features: 58
Linear Regression features: ['num__G1', 'num__G2']


,num__age,num__Medu,num__Fedu,num__traveltime,num__studytime,num__failures,num__famrel,num__freetime,num__goout,num__Dalc,...,cat__activities_yes,cat__nursery_no,cat__nursery_yes,cat__higher_no,cat__higher_yes,cat__internet_no,cat__internet_yes,cat__romantic_no,cat__romantic_yes,G3
0,0.987932,-0.437418,-0.256337,-0.754310,1.352962,-0.382133,0.068136,-0.186680,-0.196270,-0.539674,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,18
1,-0.629534,1.350140,1.591423,-0.754310,0.121054,-0.382133,0.068136,0.753970,1.501468,3.579309,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,12
2,0.987932,0.456361,-0.256337,-0.754310,1.352962,-0.382133,1.108214,-0.186680,-1.045139,-0.539674,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,12
3,0.179199,-0.437418,-1.180216,-0.754310,-1.110853,-0.382133,0.068136,0.753970,-1.045139,0.490072,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,13
4,0.179199,-1.331196,-1.180216,0.555011,-1.110853,-0.382133,-0.971942,1.694619,1.501468,0.490072,...,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,10


Step 6 – Define the four selected models

In [6]:
selected_models = {

    "Linear Regression": {
        "model": LinearRegression(),
        "features": grade_features,
        "configuration": "LR-2: G1 and G2"
    },

    "Ridge Regression": {
        "model": Ridge(alpha=10.0),
        "features": X_all.columns.tolist(),
        "configuration": "Ridge-4: alpha=10"
    },

    "Random Forest": {
        "model": RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=4,
            random_state=42,
            n_jobs=-1
        ),
        "features": X_all.columns.tolist(),
        "configuration": "RF-5"
    },

    "Gradient Boosting": {
        "model": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=2,
            min_samples_leaf=2,
            random_state=42
        ),
        "features": X_all.columns.tolist(),
        "configuration": "GB-3"
    }
}

model_setup_df = pd.DataFrame([
    {
        "Model": name,
        "Configuration": setup["configuration"],
        "Features": len(setup["features"])
    }
    for name, setup in selected_models.items()
])

display(model_setup_df)

,Model,Configuration,Features
0,Linear Regression,LR-2: G1 and G2,2
1,Ridge Regression,Ridge-4: alpha=10,58
2,Random Forest,RF-5,58
3,Gradient Boosting,GB-3,58


Step 7 – Configure identical cross-validation

In [7]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

print("Cross-validation configured!")

Cross-validation configured!


Step 8 – Retrain and compare all four models

In [8]:
comparison_results = []
fold_results = []
oof_predictions = {}

actual = y_train.to_numpy()
actual_binary = (actual >= 10).astype(int)

for name, setup in selected_models.items():

    print(f"Evaluating {name}...")

    X_model = X_all[setup["features"]]
    estimator = setup["model"]

    scores = cross_validate(
        estimator,
        X_model,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        error_score="raise",
        n_jobs=1
    )

    predictions = cross_val_predict(
        estimator,
        X_model,
        y_train,
        cv=cv,
        n_jobs=1
    )

    oof_predictions[name] = predictions

    rounded_predictions = np.clip(
        np.rint(predictions), 0, 20
    ).astype(int)

    predicted_binary = (
        predictions >= 10
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        actual_binary,
        predicted_binary,
        labels=[0, 1]
    ).ravel()

    train_rmse_folds = np.sqrt(
        -scores["train_MSE"]
    )

    val_rmse_folds = np.sqrt(
        -scores["test_MSE"]
    )

    train_rmse = train_rmse_folds.mean()
    val_rmse = val_rmse_folds.mean()

    comparison_results.append({

        "Model": name,
        "Configuration": setup["configuration"],
        "Features": len(setup["features"]),

        "Train MAE": -scores["train_MAE"].mean(),
        "Validation MAE": -scores["test_MAE"].mean(),

        "Train MSE": -scores["train_MSE"].mean(),
        "Validation MSE": -scores["test_MSE"].mean(),

        "Train RMSE": train_rmse,
        "Validation RMSE": val_rmse,

        "Train R2": scores["train_R2"].mean(),
        "Validation R2": scores["test_R2"].mean(),

        "RMSE Gap": val_rmse - train_rmse,
        "Validation RMSE Std": val_rmse_folds.std(),

        "Exact Grade Accuracy (%)": (
            np.mean(rounded_predictions == actual) * 100
        ),

        "Within 1 Grade (%)": (
            np.mean(
                np.abs(actual - predictions) <= 1
            ) * 100
        ),

        "Within 2 Grades (%)": (
            np.mean(
                np.abs(actual - predictions) <= 2
            ) * 100
        ),

        "Pass/Fail Accuracy (%)": (
            accuracy_score(
                actual_binary,
                predicted_binary
            ) * 100
        ),

        "Precision": precision_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Recall": recall_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0 else 0
        ),

        "F1 Score": f1_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    })

    for fold in range(5):

        fold_results.append({
            "Model": name,
            "Fold": fold + 1,

            "Train MAE": -scores["train_MAE"][fold],
            "Validation MAE": -scores["test_MAE"][fold],

            "Train MSE": -scores["train_MSE"][fold],
            "Validation MSE": -scores["test_MSE"][fold],

            "Train RMSE": train_rmse_folds[fold],
            "Validation RMSE": val_rmse_folds[fold],

            "Train R2": scores["train_R2"][fold],
            "Validation R2": scores["test_R2"][fold]
        })

comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    by=["Validation RMSE", "Validation MAE"],
    ascending=[True, True]
).reset_index(drop=True)

fold_df = pd.DataFrame(fold_results)

print("\nAll four models evaluated successfully!")
display(comparison_df.round(4))

Evaluating Linear Regression...
Evaluating Ridge Regression...
Evaluating Random Forest...
Evaluating Gradient Boosting...

All four models evaluated successfully!


,Model,Configuration,Features,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,...,Within 2 Grades (%),Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score,TN,FP,FN,TP
0,Linear Regression,LR-2: G1 and G2,2,0.8058,0.8163,1.6440,1.6889,1.2798,1.2720,0.8441,...,95.5684,87.8613,0.9818,0.8710,0.9176,0.9231,78,7,56,378
1,Random Forest,RF-5,58,0.5303,0.8518,0.7575,1.7397,0.8683,1.2993,0.9281,...,93.6416,89.7881,0.9751,0.9009,0.8824,0.9365,75,10,43,391
2,Gradient Boosting,GB-3,58,0.6492,0.8577,0.8988,1.7709,0.9460,1.3014,0.9148,...,92.6782,90.9441,0.9685,0.9217,0.8471,0.9445,72,13,34,400
3,Ridge Regression,Ridge-4: alpha=10,58,0.7868,0.8698,1.4651,1.8275,1.2080,1.3237,0.8611,...,93.8343,87.6686,0.9793,0.8710,0.9059,0.9220,77,8,56,378


Step 9 – Compare regression performance

In [9]:
regression_columns = [
    "Model",
    "Features",
    "Train MAE",
    "Validation MAE",
    "Train MSE",
    "Validation MSE",
    "Train RMSE",
    "Validation RMSE",
    "Train R2",
    "Validation R2",
    "RMSE Gap"
]

print("=" * 70)
print("FINAL MODEL COMPARISON – REGRESSION METRICS")
print("=" * 70)

display(
    comparison_df[regression_columns].round(4)
)

FINAL MODEL COMPARISON – REGRESSION METRICS


,Model,Features,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap
0,Linear Regression,2,0.8058,0.8163,1.6440,1.6889,1.2798,1.2720,0.8441,0.8417,-0.0078
1,Random Forest,58,0.5303,0.8518,0.7575,1.7397,0.8683,1.2993,0.9281,0.8323,0.4310
2,Gradient Boosting,58,0.6492,0.8577,0.8988,1.7709,0.9460,1.3014,0.9148,0.8315,0.3554
3,Ridge Regression,58,0.7868,0.8698,1.4651,1.8275,1.2080,1.3237,0.8611,0.8284,0.1157


Step 10 – Compare grade prediction accuracy

In [10]:
grade_columns = [
    "Model",
    "Exact Grade Accuracy (%)",
    "Within 1 Grade (%)",
    "Within 2 Grades (%)"
]

print("=" * 70)
print("FINAL MODEL COMPARISON – GRADE ACCURACY")
print("=" * 70)

display(
    comparison_df[grade_columns].round(2)
)

FINAL MODEL COMPARISON – GRADE ACCURACY


,Model,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%)
0,Linear Regression,42.20,77.84,95.57
1,Random Forest,42.20,73.03,93.64
2,Gradient Boosting,41.23,73.60,92.68
3,Ridge Regression,38.54,70.52,93.83


Step 11 – Compare pass/fail classification

In [11]:
classification_columns = [
    "Model",
    "Pass/Fail Accuracy (%)",
    "Precision",
    "Recall",
    "Specificity",
    "F1 Score"
]

print("=" * 70)
print("FINAL MODEL COMPARISON – PASS/FAIL METRICS")
print("=" * 70)

display(
    comparison_df[classification_columns].round(4)
)

FINAL MODEL COMPARISON – PASS/FAIL METRICS


,Model,Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score
0,Linear Regression,87.8613,0.9818,0.8710,0.9176,0.9231
1,Random Forest,89.7881,0.9751,0.9009,0.8824,0.9365
2,Gradient Boosting,90.9441,0.9685,0.9217,0.8471,0.9445
3,Ridge Regression,87.6686,0.9793,0.8710,0.9059,0.9220


Step 12 – Compare confusion matrix counts

In [12]:
print("=" * 70)
print("FINAL MODEL COMPARISON – CONFUSION MATRIX")
print("=" * 70)

display(
    comparison_df[
        ["Model", "TN", "FP", "FN", "TP"]
    ]
)

FINAL MODEL COMPARISON – CONFUSION MATRIX


,Model,TN,FP,FN,TP
0,Linear Regression,78,7,56,378
1,Random Forest,75,10,43,391
2,Gradient Boosting,72,13,34,400
3,Ridge Regression,77,8,56,378


In [13]:
confusion_totals = (
    comparison_df[["TN", "FP", "FN", "TP"]]
    .sum(axis=1)
)

assert (confusion_totals == len(y_train)).all()

print("All confusion matrix totals verified!")

All confusion matrix totals verified!


Step 13 – Compare overfitting and generalization

In [14]:
baseline_scores = cross_validate(
    DummyRegressor(strategy="mean"),
    X_all,
    y_train,
    cv=cv,
    scoring=scoring
)

baseline_rmse = np.sqrt(
    -baseline_scores["test_MSE"]
).mean()

def assess_fitting(row):

    if row["Validation RMSE"] >= baseline_rmse:
        return "Weak performance / possible underfitting"

    gap_ratio = (
        row["RMSE Gap"] / row["Train RMSE"]
        if row["Train RMSE"] > 0
        else np.inf
    )

    if gap_ratio > 0.30:
        return "Possible overfitting"

    return "Reasonable generalization"

comparison_df["Fitting Assessment"] = (
    comparison_df.apply(assess_fitting, axis=1)
)

print(f"Dummy Baseline RMSE: {baseline_rmse:.4f}")

display(
    comparison_df[
        [
            "Model",
            "Train RMSE",
            "Validation RMSE",
            "RMSE Gap",
            "Validation RMSE Std",
            "Fitting Assessment"
        ]
    ].round(4)
)

Dummy Baseline RMSE: 3.2326


,Model,Train RMSE,Validation RMSE,RMSE Gap,Validation RMSE Std,Fitting Assessment
0,Linear Regression,1.2798,1.2720,-0.0078,0.2662,Reasonable generalization
1,Random Forest,0.8683,1.2993,0.4310,0.2271,Possible overfitting
2,Gradient Boosting,0.9460,1.3014,0.3554,0.2779,Possible overfitting
3,Ridge Regression,1.2080,1.3237,0.1157,0.2743,Reasonable generalization


Step 14 – Compare five-fold performance

In [15]:
print("=" * 70)
print("FINAL MODEL COMPARISON – FOLD PERFORMANCE")
print("=" * 70)

display(
    fold_df.sort_values(
        ["Model", "Fold"]
    ).round(4)
)

FINAL MODEL COMPARISON – FOLD PERFORMANCE


,Model,Fold,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2
15,Gradient Boosting,1,0.6657,0.7267,1.0068,1.0941,1.0034,1.0460,0.9090,0.8691
16,Gradient Boosting,2,0.6603,0.8562,0.9091,1.7087,0.9535,1.3072,0.9188,0.7755
17,Gradient Boosting,3,0.6518,0.7999,0.9123,1.5991,0.9551,1.2646,0.9085,0.8746
18,Gradient Boosting,4,0.5977,1.1208,0.6852,3.3046,0.8278,1.8179,0.9283,0.7612
19,Gradient Boosting,5,0.6704,0.7849,0.9807,1.1482,0.9903,1.0715,0.9093,0.8774
0,Linear Regression,1,0.8137,0.8045,1.7311,1.3519,1.3157,1.1627,0.8435,0.8382
1,Linear Regression,2,0.8306,0.7527,1.7929,1.1076,1.3390,1.0524,0.8398,0.8545
2,Linear Regression,3,0.8152,0.7522,1.6674,1.5871,1.2913,1.2598,0.8327,0.8755
3,Linear Regression,4,0.7512,1.0145,1.2702,3.1897,1.1270,1.7860,0.8671,0.7695
4,Linear Regression,5,0.8184,0.7574,1.7583,1.2084,1.3260,1.0993,0.8374,0.8709


Step 15 – Rank the models and select the winner

In [16]:
final_comparison = comparison_df.copy()

final_comparison.insert(
    0,
    "Rank",
    np.arange(1, len(final_comparison) + 1)
)

print("=" * 70)
print("FINAL PORTUGUESE MODEL RANKING")
print("=" * 70)

display(
    final_comparison[
        [
            "Rank",
            "Model",
            "Configuration",
            "Features",
            "Validation RMSE",
            "Validation MAE",
            "Validation R2",
            "Exact Grade Accuracy (%)",
            "Pass/Fail Accuracy (%)",
            "RMSE Gap",
            "Fitting Assessment"
        ]
    ].round(4)
)

best_result = final_comparison.iloc[0]
best_name = best_result["Model"]
best_setup = selected_models[best_name]

print("\n" + "=" * 65)
print("FINAL SELECTED PORTUGUESE MODEL")
print("=" * 65)

print("Model:", best_name)
print("Configuration:", best_setup["configuration"])
print("Features:", best_setup["features"])

print("\nREGRESSION PERFORMANCE")
print(f"Validation RMSE : {best_result['Validation RMSE']:.4f}")
print(f"Validation MAE  : {best_result['Validation MAE']:.4f}")
print(f"Validation R²   : {best_result['Validation R2']:.4f}")
print(f"RMSE Gap        : {best_result['RMSE Gap']:.4f}")

print("\nGRADE ACCURACY")
print(f"Exact Grade Accuracy: {best_result['Exact Grade Accuracy (%)']:.2f}%")
print(f"Within ±1 Grade    : {best_result['Within 1 Grade (%)']:.2f}%")
print(f"Within ±2 Grades   : {best_result['Within 2 Grades (%)']:.2f}%")

print("\nPASS/FAIL PERFORMANCE")
print(f"Accuracy : {best_result['Pass/Fail Accuracy (%)']:.2f}%")
print(f"Precision: {best_result['Precision']:.4f}")
print(f"Recall   : {best_result['Recall']:.4f}")
print(f"F1 Score : {best_result['F1 Score']:.4f}")

print("\nFITTING ASSESSMENT")
print(best_result["Fitting Assessment"])

FINAL PORTUGUESE MODEL RANKING


,Rank,Model,Configuration,Features,Validation RMSE,Validation MAE,Validation R2,Exact Grade Accuracy (%),Pass/Fail Accuracy (%),RMSE Gap,Fitting Assessment
0,1,Linear Regression,LR-2: G1 and G2,2,1.2720,0.8163,0.8417,42.1965,87.8613,-0.0078,Reasonable generalization
1,2,Random Forest,RF-5,58,1.2993,0.8518,0.8323,42.1965,89.7881,0.4310,Possible overfitting
2,3,Gradient Boosting,GB-3,58,1.3014,0.8577,0.8315,41.2331,90.9441,0.3554,Possible overfitting
3,4,Ridge Regression,Ridge-4: alpha=10,58,1.3237,0.8698,0.8284,38.5356,87.6686,0.1157,Reasonable generalization



FINAL SELECTED PORTUGUESE MODEL
Model: Linear Regression
Configuration: LR-2: G1 and G2
Features: ['num__G1', 'num__G2']

REGRESSION PERFORMANCE
Validation RMSE : 1.2720
Validation MAE  : 0.8163
Validation R²   : 0.8417
RMSE Gap        : -0.0078

GRADE ACCURACY
Exact Grade Accuracy: 42.20%
Within ±1 Grade    : 77.84%
Within ±2 Grades   : 95.57%

PASS/FAIL PERFORMANCE
Accuracy : 87.86%
Precision: 0.9818
Recall   : 0.8710
F1 Score : 0.9231

FITTING ASSESSMENT
Reasonable generalization


Step 16 – Train the selected model on all training records

In [17]:
final_model = clone(best_setup["model"])

final_features = best_setup["features"]

X_final_train = X_all[final_features]

final_model.fit(
    X_final_train,
    y_train
)

print("Final model trained successfully!")
print("Model:", best_name)
print("Training records:", len(y_train))
print("Features:", len(final_features))

Final model trained successfully!
Model: Linear Regression
Training records: 519
Features: 2


In [18]:
# ============================================================
# FINAL PORTUGUESE MODEL SELECTION
# ============================================================

# Rank models by validation RMSE
final_comparison = comparison_df.sort_values(
    by=["Validation RMSE", "Validation MAE"],
    ascending=[True, True]
).reset_index(drop=True)

final_comparison.insert(
    0,
    "Rank",
    np.arange(1, len(final_comparison) + 1)
)

# Display all model rankings
print("=" * 70)
print("FINAL PORTUGUESE MODEL COMPARISON")
print("=" * 70)

display(
    final_comparison[
        [
            "Rank",
            "Model",
            "Configuration",
            "Features",
            "Validation RMSE",
            "Validation MAE",
            "Validation R2",
            "Exact Grade Accuracy (%)",
            "Within 1 Grade (%)",
            "Within 2 Grades (%)",
            "Pass/Fail Accuracy (%)",
            "Fitting Assessment"
        ]
    ].round(4)
)

# Select the best model
best_result = final_comparison.iloc[0]

best_name = best_result["Model"]
best_setup = selected_models[best_name]

# ============================================================
# PRINT COMPLETE WINNING MODEL DETAILS
# ============================================================

print("\n" + "=" * 65)
print("BEST PORTUGUESE FINAL MODEL CONFIGURATION")
print("=" * 65)

print("Selected Model:", best_name)
print("Configuration:", best_setup["configuration"])
print("Number of Features:", len(best_setup["features"]))

print("\nREGRESSION METRICS")

print(
    f"Training RMSE   : "
    f"{best_result['Train RMSE']:.4f}"
)

print(
    f"Validation RMSE : "
    f"{best_result['Validation RMSE']:.4f}"
)

print(
    f"Validation MAE  : "
    f"{best_result['Validation MAE']:.4f}"
)

print(
    f"Validation MSE  : "
    f"{best_result['Validation MSE']:.4f}"
)

print(
    f"Validation R²   : "
    f"{best_result['Validation R2']:.4f}"
)

print(
    f"RMSE Gap        : "
    f"{best_result['RMSE Gap']:.4f}"
)

print("\nGRADE ACCURACY")

print(
    f"Exact Accuracy  : "
    f"{best_result['Exact Grade Accuracy (%)']:.2f}%"
)

print(
    f"Within ±1 Grade : "
    f"{best_result['Within 1 Grade (%)']:.2f}%"
)

print(
    f"Within ±2 Grades: "
    f"{best_result['Within 2 Grades (%)']:.2f}%"
)

print("\nPASS/FAIL CLASSIFICATION")

print(
    f"Accuracy   : "
    f"{best_result['Pass/Fail Accuracy (%)']:.2f}%"
)

print(
    f"Precision  : "
    f"{best_result['Precision']:.4f}"
)

print(
    f"Recall     : "
    f"{best_result['Recall']:.4f}"
)

print(
    f"Specificity: "
    f"{best_result['Specificity']:.4f}"
)

print(
    f"F1 Score   : "
    f"{best_result['F1 Score']:.4f}"
)

print("\nCONFUSION MATRIX")

print("TN:", int(best_result["TN"]))
print("FP:", int(best_result["FP"]))
print("FN:", int(best_result["FN"]))
print("TP:", int(best_result["TP"]))

print("\nFITTING ASSESSMENT")

print(best_result["Fitting Assessment"])

print("\n" + "=" * 65)
print("FINAL MODEL SELECTION COMPLETED")
print("=" * 65)

print("Selected Model:", best_name)
print("Selection Criterion: Lowest Validation RMSE")

FINAL PORTUGUESE MODEL COMPARISON


,Rank,Model,Configuration,Features,Validation RMSE,Validation MAE,Validation R2,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%),Pass/Fail Accuracy (%),Fitting Assessment
0,1,Linear Regression,LR-2: G1 and G2,2,1.2720,0.8163,0.8417,42.1965,77.8420,95.5684,87.8613,Reasonable generalization
1,2,Random Forest,RF-5,58,1.2993,0.8518,0.8323,42.1965,73.0250,93.6416,89.7881,Possible overfitting
2,3,Gradient Boosting,GB-3,58,1.3014,0.8577,0.8315,41.2331,73.6031,92.6782,90.9441,Possible overfitting
3,4,Ridge Regression,Ridge-4: alpha=10,58,1.3237,0.8698,0.8284,38.5356,70.5202,93.8343,87.6686,Reasonable generalization



BEST PORTUGUESE FINAL MODEL CONFIGURATION
Selected Model: Linear Regression
Configuration: LR-2: G1 and G2
Number of Features: 2

REGRESSION METRICS
Training RMSE   : 1.2798
Validation RMSE : 1.2720
Validation MAE  : 0.8163
Validation MSE  : 1.6889
Validation R²   : 0.8417
RMSE Gap        : -0.0078

GRADE ACCURACY
Exact Accuracy  : 42.20%
Within ±1 Grade : 77.84%
Within ±2 Grades: 95.57%

PASS/FAIL CLASSIFICATION
Accuracy   : 87.86%
Precision  : 0.9818
Recall     : 0.8710
Specificity: 0.9176
F1 Score   : 0.9231

CONFUSION MATRIX
TN: 78
FP: 7
FN: 56
TP: 378

FITTING ASSESSMENT
Reasonable generalization

FINAL MODEL SELECTION COMPLETED
Selected Model: Linear Regression
Selection Criterion: Lowest Validation RMSE
